In [4]:
import pandas as pd
file_path = '../data/raw/AmazonData.xlsx'

df_products = pd.read_excel(file_path, sheet_name = 'Products')
df_reviews = pd.read_excel(file_path, sheet_name = 'Reviews')

In [9]:
# drop if there are duplicate asin
df_products = df_products.drop_duplicates(subset = 'ASIN')

In [10]:
# validate price (not null or negative)
df_products = df_products[df_products['Price'].notnull()]
df_products = df_products[df_products['Price'] > 0]

In [12]:
# validate rating (not null or out of range)
df_products = df_products[
    (df_products['Rating'].isnull()) | 
    ((df_products['Rating'] >= 1) & (df_products['Rating'] <= 5))
]

In [14]:
# validate review count (not null or negative)
df_products = df_products[
    (df_products['Review Count'].isnull()) |
    (df_products['Review Count'] >= 0)
]

In [15]:
# reset index
df_products.reset_index(drop = True, inplace = True)

In [17]:
# drop if there are duplicate reviews
df_reviews = df_reviews.drop_duplicates(subset = ['ASIN', 'Review_Body'])

In [19]:
# drop critical nulls
df_reviews = df_reviews.dropna(subset = ['Review_Body', 'Review_Title'])

In [20]:
# drop null text
df_reviews = df_reviews[df_reviews['Review_Body'] != ""]
df_reviews = df_reviews[df_reviews['Review_Title'] != ""]

In [21]:
# validate stars
df_reviews = df_reviews[
    (df_reviews['Stars'] >= 1) & (df_reviews['Stars'] <= 5)
]

In [22]:
# reset index
df_reviews.reset_index(drop = True, inplace = True)

In [28]:
# final validation
print(df_products.info())
print(df_products.describe())

<class 'pandas.core.frame.DataFrame'>
Index: 179 entries, 1 to 180
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ASIN          179 non-null    object 
 1   Title         179 non-null    object 
 2   Price         179 non-null    float64
 3   Rating        179 non-null    float64
 4   Review Count  179 non-null    float64
 5   Date          179 non-null    object 
 6   URL           179 non-null    object 
dtypes: float64(3), object(4)
memory usage: 11.2+ KB
None
            Price      Rating  Review Count
count  179.000000  179.000000    179.000000
mean    17.615419    4.615084    370.178771
std      2.597689    0.396539   1270.620158
min      7.910000    2.000000      1.000000
25%     15.055000    4.500000      5.000000
50%     17.410000    4.700000     26.000000
75%     19.800000    4.800000    157.500000
max     24.750000    5.000000   9909.000000


In [29]:
print(df_reviews.info())
print(df_reviews.describe())

<class 'pandas.core.frame.DataFrame'>
Index: 825 entries, 8 to 832
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ASIN               825 non-null    object
 1   Stars              825 non-null    int64 
 2   Review_Title       825 non-null    object
 3   Verified_Purchase  825 non-null    object
 4   Review_Date        825 non-null    object
 5   Review_Body        825 non-null    object
 6   Extraction_Date    825 non-null    object
dtypes: int64(1), object(6)
memory usage: 51.6+ KB
None
            Stars
count  825.000000
mean     4.626667
std      0.890220
min      1.000000
25%      5.000000
50%      5.000000
75%      5.000000
max      5.000000


In [30]:
# final duplicates
print(df_products.duplicated(subset = 'ASIN').sum())
print(df_reviews.duplicated(subset = ['ASIN', 'Review_Body']).sum())

0
0


In [26]:
# delete outliers
q99 = df_products['Price'].quantile(0.99)
df_products = df_products[df_products['Price'] <= q99]
df_reviews = df_reviews[df_reviews['ASIN'].isin(df_products['ASIN'])]

In [27]:
df_products.shape
df_reviews.shape

# check for reviews with ASIN
df_reviews['ASIN'].isin(df_products['ASIN']).all()

np.True_

In [31]:
# save clean dataset
with pd.ExcelWriter('../data/processed/AmazonData_clean.xlsx', engine = 'openpyxl') as writer:
    df_products.to_excel(writer, sheet_name = 'Products', index = False)
    df_reviews.to_excel(writer, sheet_name = 'Reviews', index = False)